In [9]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
#Load environment variables
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [12]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)

# Make a request
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message.content[0].text


In [13]:
import json

def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompt 
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON
    each representing task that requires Python, JSON, or a Regex to complete.
    
    Example output:
    ```json
    [
        {
            "task": "Description of the task",
        },
        ...additional
    ] 
    ``` 

    * Focus on tasks that can be solved writing a single Python function, a single JSON object,
    * Focus on tasks that do not require writing much code

    Please generate 3 objects
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [16]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [17]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    
    {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [18]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }


In [19]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case for each test case, then returns the results"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    return results

In [20]:
with open("dataset.json") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

In [21]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Parser\n\nHere's a Python function that parses AWS S3 bucket URIs:\n\n```python\ndef parse_s3_uri(uri: str) -> dict:\n    \"\"\"\n    Parse an AWS S3 bucket URI and return bucket and key components.\n    \n    Args:\n        uri (str): S3 URI in the format 's3://bucket-name/key/path'\n    \n    Returns:\n        dict: Dictionary with 'bucket' and 'key' fields\n        \n    Raises:\n        ValueError: If URI format is invalid\n    \"\"\"\n    # Remove leading/trailing whitespace\n    uri = uri.strip()\n    \n    # Check if URI starts with s3://\n    if not uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI: must start with 's3://', got '{uri}'\")\n    \n    # Remove the s3:// prefix\n    uri_without_protocol = uri[5:]\n    \n    # Split by the first forward slash to separate bucket from key\n    if '/' not in uri_without_protocol:\n        # Only bucket, no key\n        return {\n            'bucket': uri_without_protocol,\n         